In [2]:
# --- Third-party libraries: numerical + plotting ---
import numpy as np                               # numerical computing (arrays, arange, hstack, etc.)

# --- Deep learning framework ---
import tensorflow as tf                          # core TensorFlow library
import tensorflow.keras as keras                 # high-level neural network API

# --- Evaluation imports ---
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

2026-04-13 21:07:45.163065: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### 1. MNIST laden

In [3]:
tf.random.set_seed(42)
np.random.seed(42)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Augabe der Shape: Anzahl Bilder, Höhe, Breite 
print("Train shape:", x_train.shape)

Train shape: (60000, 28, 28)


### 2. Preprocessing

#### 2.1 Normalisierung

In [4]:
# Werte vom ursprünglichen Bereich 0–255 in den Bereich 0–1 umwandeln.
x_train = x_train / 255.0
x_test = x_test / 255.0

#### 2.2 Kanal-Dimension

In [5]:
# CNN um eine Kanal-Dimension erweitern um die geforderte Form (height, width, channels) zu erreichen
# Shape: (Anzahl Bilder, Höhe, Breite) zu (Anzahl Bilder, Höhe, Breite, Kanäle)

x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

#### 2.3 Daten Augmentation

In [6]:
# Daten leicht rotieren und verschieben, um Variationen zu simulieren
data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomTranslation(0.05, 0.05)
])

### 3. CNN Modell

In [7]:
model = keras.Sequential([

    # ------------------------------------------------------------
    # Input layer
    # Erwartet Bilder der Form (28, 28, 1)
    # ------------------------------------------------------------
    keras.layers.Input(shape=(28,28,1)),

    # ------------------------------------------------------------
    # Implementierung der Data Augmentation während des Trainings
    # um Robustheit gegenüber Rotation und Verschiebung zu lernen.
    # ------------------------------------------------------------
    data_augmentation,


    # ------------------------------------------------------------
    # Block 1: Low-level Feature Extraction
    # Lernen einfacher Muster wie Kanten und Linien
    # ------------------------------------------------------------

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.MaxPooling2D((2,2)),

    keras.layers.Dropout(0.2),


    # ------------------------------------------------------------
    # Block 2: High-level Feature Extraction
    # Lernen komplexerer Strukturen
    # ------------------------------------------------------------

    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),
    
    keras.layers.Conv2D(64, (3,3), padding="same"),
    keras.layers.BatchNormalization(),
    keras.layers.Activation("relu"),

    keras.layers.MaxPooling2D((2,2)),

    keras.layers.Dropout(0.3),


    # ------------------------------------------------------------
    # Classification Head
    # Umwandeln von Feature Maps in Klassenentscheidung
    # ------------------------------------------------------------

    keras.layers.Flatten(),

    keras.layers.Dense(128, activation="relu"),

    keras.layers.Dropout(0.4),

    keras.layers.Dense(10, activation="softmax")
])


### 4. Kompilieren

In [8]:
# Modell kompilieren
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 28, 28, 64)        640       
                                                                 
 batch_normalization (Batch  (None, 28, 28, 64)        256       
 Normalization)                                                  
                                                                 
 activation (Activation)     (None, 28, 28, 64)        0         
                                                                 
 conv2d_1 (Conv2D)           (None, 28, 28, 64)        36928     
                                                                 
 batch_normalization_1 (Bat  (None, 28, 28, 64)        256       
 chNormalization)                                     

### 5. Training

In [9]:
# Abbruchkriterium festlegen um Overfitting zu verhindern und Trainingszeit zu sparen
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [10]:
# Modell auf Trainingsdaten trainieren und Validierung überwachen
model.fit(
    x_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/20
1688/1688 [==============================] - 165s 96ms/step - loss: 0.4003 - accuracy: 0.8748 - val_loss: 0.0568 - val_accuracy: 0.9835
Epoch 2/20
1688/1688 [==============================] - 160s 95ms/step - loss: 0.1682 - accuracy: 0.9500 - val_loss: 0.0429 - val_accuracy: 0.9887
Epoch 3/20
1688/1688 [==============================] - 161s 96ms/step - loss: 0.1342 - accuracy: 0.9608 - val_loss: 0.0400 - val_accuracy: 0.9905
Epoch 4/20
1688/1688 [==============================] - 151s 89ms/step - loss: 0.1090 - accuracy: 0.9675 - val_loss: 0.0353 - val_accuracy: 0.9910
Epoch 5/20
1688/1688 [==============================] - 152s 90ms/step - loss: 0.0923 - accuracy: 0.9731 - val_loss: 0.0306 - val_accuracy: 0.9913
Epoch 6/20
1688/1688 [==============================] - 155s 92ms/step - loss: 0.0820 - accuracy: 0.9766 - val_loss: 0.0323 - val_accuracy: 0.9923
Epoch 7/20
1688/1688 [==============================] - 155s 92ms/step - loss: 0.0715 - accuracy: 0.9794 - val_loss: 0

### 6. Modell speichern

In [11]:
model.save("CNN_Vincent.keras")

### 7. Evaluation

In [16]:
# Vorhersagen
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = y_test

# Klassische Metriken
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
precision = precision_score(y_true, y_pred, average="macro")
recall = recall_score(y_true, y_pred, average="macro")
f1 = f1_score(y_true, y_pred, average="macro")

# Confusion Matrix für FPR & FNR
cm = confusion_matrix(y_true, y_pred)
FP = cm.sum(axis=0) - np.diag(cm)
FN = cm.sum(axis=1) - np.diag(cm)
TP = np.diag(cm)
TN = cm.sum() - (FP + FN + TP)

FPR = np.mean(FP / (FP + TN + 1e-10))
FNR = np.mean(FN / (FN + TP + 1e-10))

# Report
report = classification_report(y_true, y_pred, digits=4)

# Ausgabe
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"FPR: {FPR:.4f}")
print(f"FNR: {FNR:.4f}")
print("\nReport: \n", report)

Accuracy: 0.9953
Precision: 0.9953
Recall: 0.9952
F1-Score: 0.9952
FPR: 0.0005
FNR: 0.0048

Report: 
               precision    recall  f1-score   support

           0     0.9969    0.9959    0.9964       980
           1     0.9947    0.9982    0.9965      1135
           2     0.9971    0.9971    0.9971      1032
           3     0.9960    0.9980    0.9970      1010
           4     0.9909    0.9990    0.9949       982
           5     0.9933    0.9955    0.9944       892
           6     0.9947    0.9854    0.9900       958
           7     0.9961    0.9951    0.9956      1028
           8     0.9949    0.9949    0.9949       974
           9     0.9980    0.9931    0.9955      1009

    accuracy                         0.9953     10000
   macro avg     0.9953    0.9952    0.9952     10000
weighted avg     0.9953    0.9953    0.9953     10000



### 8. Nachweis: Grid-Search